# Validation Against Experimental Data

This advanced tutorial demonstrates how to compare DPF2 simulations against experimental data from real Dense Plasma Focus devices.

## Learning Objectives

After completing this notebook, you will be able to:

1. Load and process experimental DPF data
2. Set up and run dpf2 simulations matching experimental conditions
3. Compare simulation results with experimental measurements
4. Perform statistical analysis of model-data agreement
5. Identify sources of discrepancy and model limitations

## 1. Introduction

Validation is a critical step in computational physics. We need to answer:

- **Does our model capture the essential physics?**
- **How well do simulations match reality?**
- **What are the model limitations?**

### Validation Hierarchy

```
┌─────────────────────────────────────────────────────────────────┐
│                    VALIDATION HIERARCHY                         │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  Level 1: Unit Tests          - Code correctness               │
│  Level 2: Verification        - Solve equations correctly      │
│  Level 3: Validation          - Equations represent physics    │
│  Level 4: Prediction          - Extrapolate to new regimes     │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

This notebook focuses on **Level 3: Validation**.

## 2. Setup and Imports

In [ ]:
# Standard imports
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json

# Statistical analysis
from scipy import stats
from scipy.interpolate import interp1d
from scipy.optimize import minimize

# DPF2 imports
try:
    from dpf2.circuit_solver import run_circuit_simulation
    from dpf2.circuit_config import CircuitConfig
    from dpf2.scaling_laws import lee_scaling, neutron_yield_scaling
    from dpf2.validation_suite import ValidationMetrics
    DPF2_AVAILABLE = True
except ImportError:
    print("Note: Some dpf2 modules not available. Using fallback implementations.")
    DPF2_AVAILABLE = False

# Configure plotting
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['font.size'] = 12
plt.style.use('seaborn-v0_8-whitegrid')

# Set random seed for reproducibility
np.random.seed(42)

print("Setup complete!")

## 3. Loading Experimental Data

We'll work with synthetic data that mimics real DPF experimental measurements.

### 3.1 Data Structure

Experimental DPF data typically includes:
- Current waveform (Rogowski coil)
- Voltage waveform (resistive divider)
- Neutron yield (activation or scintillator)
- X-ray signals (PIN diodes, filtered)
- Pinch time and radial compression

In [ ]:
def generate_synthetic_experimental_data(n_shots=50):
    """
    Generate synthetic experimental data mimicking a real DPF device.
    
    This simulates a medium-sized DPF with:
    - 15 kJ stored energy
    - 500 kA peak current
    - ~10^8 neutron yield
    
    Parameters
    ----------
    n_shots : int
        Number of experimental shots to simulate
        
    Returns
    -------
    dict
        Dictionary containing experimental data
    """
    # Device parameters (with shot-to-shot variation)
    V0_mean = 25e3  # 25 kV charging voltage
    V0_std = 0.5e3  # 0.5 kV variation
    
    # Fill pressure variation
    p_mean = 4.0  # mbar
    p_std = 0.2   # mbar
    
    shots = []
    for i in range(n_shots):
        # Random variations
        V0 = np.random.normal(V0_mean, V0_std)
        pressure = np.random.normal(p_mean, p_std)
        
        # Peak current (with noise)
        # Empirical: I_peak ≈ 20 * V0^0.5 for this device class
        I_peak_ideal = 500e3 * (V0 / V0_mean)**0.5
        I_peak = I_peak_ideal * np.random.normal(1.0, 0.03)
        
        # Current rise time
        t_rise_ideal = 1.5e-6  # 1.5 μs
        t_rise = t_rise_ideal * np.random.normal(1.0, 0.05)
        
        # Pinch time (from current start)
        t_pinch_ideal = 1.8e-6  # 1.8 μs
        t_pinch = t_pinch_ideal * np.random.normal(1.0, 0.08)
        
        # Neutron yield (I^4 scaling with large scatter)
        Y_n_ideal = 1e8 * (I_peak / 500e3)**4
        # Log-normal distribution for yield (common in DPF)
        Y_n = Y_n_ideal * np.random.lognormal(0, 0.5)
        
        # dI/dt at pinch (voltage spike indicator)
        dIdt_pinch = -5e12 * np.random.normal(1.0, 0.15)  # -5 kA/ns
        
        shots.append({
            'shot_number': i + 1,
            'V0_kV': V0 / 1e3,
            'pressure_mbar': pressure,
            'I_peak_kA': I_peak / 1e3,
            't_rise_us': t_rise * 1e6,
            't_pinch_us': t_pinch * 1e6,
            'Y_n': Y_n,
            'dIdt_pinch_kA_ns': dIdt_pinch / 1e12
        })
    
    return {
        'device': 'Synthetic DPF',
        'description': 'Synthetic data mimicking 15 kJ DPF',
        'parameters': {
            'C_uF': 48,
            'L_nH': 40,
            'R_mohm': 2.5,
            'anode_radius_cm': 1.2,
            'anode_length_cm': 16,
            'cathode_radius_cm': 3.5,
            'fill_gas': 'D2'
        },
        'shots': shots
    }

# Generate data
exp_data = generate_synthetic_experimental_data(n_shots=100)

print(f"Device: {exp_data['device']}")
print(f"Number of shots: {len(exp_data['shots'])}")
print(f"\nDevice parameters:")
for key, value in exp_data['parameters'].items():
    print(f"  {key}: {value}")

### 3.2 Data Exploration

In [ ]:
# Convert to numpy arrays for analysis
shots = exp_data['shots']
V0 = np.array([s['V0_kV'] for s in shots])
I_peak = np.array([s['I_peak_kA'] for s in shots])
Y_n = np.array([s['Y_n'] for s in shots])
t_pinch = np.array([s['t_pinch_us'] for s in shots])
pressure = np.array([s['pressure_mbar'] for s in shots])

# Summary statistics
print("Experimental Data Summary")
print("=" * 50)
print(f"Charging voltage: {V0.mean():.1f} ± {V0.std():.1f} kV")
print(f"Peak current:     {I_peak.mean():.0f} ± {I_peak.std():.0f} kA")
print(f"Neutron yield:    {Y_n.mean():.2e} ± {Y_n.std():.2e}")
print(f"Pinch time:       {t_pinch.mean():.2f} ± {t_pinch.std():.2f} μs")
print(f"Fill pressure:    {pressure.mean():.2f} ± {pressure.std():.2f} mbar")

In [ ]:
# Visualize experimental data distributions
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

# Voltage distribution
axes[0, 0].hist(V0, bins=15, edgecolor='black', alpha=0.7)
axes[0, 0].axvline(V0.mean(), color='red', linestyle='--', label=f'Mean: {V0.mean():.1f} kV')
axes[0, 0].set_xlabel('Charging Voltage (kV)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Voltage Distribution')
axes[0, 0].legend()

# Current distribution
axes[0, 1].hist(I_peak, bins=15, edgecolor='black', alpha=0.7, color='orange')
axes[0, 1].axvline(I_peak.mean(), color='red', linestyle='--', label=f'Mean: {I_peak.mean():.0f} kA')
axes[0, 1].set_xlabel('Peak Current (kA)')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Peak Current Distribution')
axes[0, 1].legend()

# Neutron yield distribution (log scale)
axes[0, 2].hist(np.log10(Y_n), bins=15, edgecolor='black', alpha=0.7, color='green')
axes[0, 2].axvline(np.log10(Y_n.mean()), color='red', linestyle='--', label=f'Mean: {Y_n.mean():.1e}')
axes[0, 2].set_xlabel('log₁₀(Neutron Yield)')
axes[0, 2].set_ylabel('Count')
axes[0, 2].set_title('Neutron Yield Distribution')
axes[0, 2].legend()

# I_peak vs V0 correlation
axes[1, 0].scatter(V0, I_peak, alpha=0.6)
z = np.polyfit(V0, I_peak, 1)
p = np.poly1d(z)
V0_fit = np.linspace(V0.min(), V0.max(), 100)
axes[1, 0].plot(V0_fit, p(V0_fit), 'r-', label=f'Linear fit')
axes[1, 0].set_xlabel('Charging Voltage (kV)')
axes[1, 0].set_ylabel('Peak Current (kA)')
axes[1, 0].set_title('Current vs Voltage')
r_IV = np.corrcoef(V0, I_peak)[0, 1]
axes[1, 0].text(0.05, 0.95, f'r = {r_IV:.3f}', transform=axes[1, 0].transAxes, verticalalignment='top')

# Yield vs I_peak (log scale)
axes[1, 1].scatter(I_peak, Y_n, alpha=0.6, color='green')
axes[1, 1].set_yscale('log')
# Fit I^4 scaling
I_fit = np.linspace(I_peak.min(), I_peak.max(), 100)
Y_fit = (Y_n.mean() / (I_peak.mean()**4)) * I_fit**4
axes[1, 1].plot(I_fit, Y_fit, 'r-', label='I⁴ scaling')
axes[1, 1].set_xlabel('Peak Current (kA)')
axes[1, 1].set_ylabel('Neutron Yield')
axes[1, 1].set_title('Yield vs Current')
axes[1, 1].legend()

# Pinch time vs pressure
axes[1, 2].scatter(pressure, t_pinch, alpha=0.6, color='purple')
axes[1, 2].set_xlabel('Fill Pressure (mbar)')
axes[1, 2].set_ylabel('Pinch Time (μs)')
axes[1, 2].set_title('Pinch Time vs Pressure')
r_pt = np.corrcoef(pressure, t_pinch)[0, 1]
axes[1, 2].text(0.05, 0.95, f'r = {r_pt:.3f}', transform=axes[1, 2].transAxes, verticalalignment='top')

plt.tight_layout()
plt.show()

## 4. Setting Up the Simulation

Now we'll configure dpf2 to match the experimental device parameters.

In [ ]:
def create_simulation_config(device_params, shot_params):
    """
    Create a dpf2 simulation configuration matching experimental conditions.
    
    Parameters
    ----------
    device_params : dict
        Device parameters (capacitance, inductance, geometry)
    shot_params : dict
        Shot-specific parameters (voltage, pressure)
        
    Returns
    -------
    dict
        Configuration dictionary for dpf2
    """
    config = {
        'circuit': {
            'C_uF': device_params['C_uF'],
            'L_nH': device_params['L_nH'],
            'R_mohm': device_params['R_mohm'],
            'V0_kV': shot_params['V0_kV']
        },
        'geometry': {
            'anode_radius_cm': device_params['anode_radius_cm'],
            'anode_length_cm': device_params['anode_length_cm'],
            'cathode_radius_cm': device_params['cathode_radius_cm']
        },
        'gas': {
            'species': device_params['fill_gas'],
            'pressure_mbar': shot_params['pressure_mbar']
        },
        'simulation': {
            't_end_us': 5.0,
            'dt_ns': 1.0
        }
    }
    return config

# Create config for mean experimental conditions
mean_shot = {
    'V0_kV': V0.mean(),
    'pressure_mbar': pressure.mean()
}

config = create_simulation_config(exp_data['parameters'], mean_shot)
print("Simulation Configuration:")
print(json.dumps(config, indent=2))

## 5. Running the Simulation

We'll use a simplified model for demonstration that captures the key physics.

In [ ]:
def run_dpf_simulation(config):
    """
    Run a simplified DPF simulation.
    
    This is a surrogate model that captures key DPF behavior:
    - RLC circuit dynamics
    - Lee model-like plasma dynamics
    - Neutron yield scaling
    
    Parameters
    ----------
    config : dict
        Simulation configuration
        
    Returns
    -------
    dict
        Simulation results
    """
    # Extract parameters
    C = config['circuit']['C_uF'] * 1e-6
    L = config['circuit']['L_nH'] * 1e-9
    R = config['circuit']['R_mohm'] * 1e-3
    V0 = config['circuit']['V0_kV'] * 1e3
    
    a_anode = config['geometry']['anode_radius_cm'] * 1e-2
    l_anode = config['geometry']['anode_length_cm'] * 1e-2
    b_cathode = config['geometry']['cathode_radius_cm'] * 1e-2
    
    p = config['gas']['pressure_mbar']
    
    # Time array
    t_end = config['simulation']['t_end_us'] * 1e-6
    dt = config['simulation']['dt_ns'] * 1e-9
    t = np.arange(0, t_end, dt)
    
    # Circuit parameters
    omega_0 = 1.0 / np.sqrt(L * C)
    alpha = R / (2 * L)
    omega_d = np.sqrt(max(0, omega_0**2 - alpha**2))
    
    # RLC current (analytical)
    if omega_d > 0:
        I = (V0 / (L * omega_d)) * np.exp(-alpha * t) * np.sin(omega_d * t)
    else:
        I = (V0 / L) * t * np.exp(-alpha * t)
    
    # Peak current
    I_peak = np.max(I)
    t_peak = t[np.argmax(I)]
    
    # Simplified rundown time (Lee model approximation)
    # t_rundown ∝ (ρ * l * a)^0.5 / (μ0 * I)^0.5
    rho_gas = p * 1e2 * 4e-3 / (8.314 * 300)  # D2 mass density at room temp
    mu_0 = 4 * np.pi * 1e-7
    
    t_rundown = 1.5e-6 * (p / 4.0)**0.3  # Empirical scaling
    t_pinch = t_rundown * 1.2
    
    # Current at pinch
    if t_pinch < t[-1]:
        idx_pinch = int(t_pinch / dt)
        I_pinch = I[min(idx_pinch, len(I)-1)]
    else:
        I_pinch = I_peak * 0.8
    
    # Neutron yield (simplified I^4 scaling with focus factor)
    # Y_n = k * I_pinch^4 * f(geometry) * f(pressure)
    focus_factor = (b_cathode / a_anode) / 3.0  # Cathode/anode ratio effect
    pressure_factor = np.exp(-((p - 4.0)/2.0)**2)  # Optimal around 4 mbar
    
    k_yield = 2e-17  # Empirical constant
    Y_n = k_yield * I_pinch**4 * focus_factor * pressure_factor
    
    # dI/dt at pinch
    if idx_pinch < len(I) - 1:
        dIdt_pinch = (I[min(idx_pinch+1, len(I)-1)] - I[max(idx_pinch-1, 0)]) / (2 * dt)
    else:
        dIdt_pinch = 0
    
    return {
        't': t,
        'I': I,
        'I_peak_kA': I_peak / 1e3,
        't_rise_us': t_peak * 1e6,
        't_pinch_us': t_pinch * 1e6,
        'I_pinch_kA': I_pinch / 1e3,
        'Y_n': Y_n,
        'dIdt_pinch_kA_ns': dIdt_pinch / 1e12
    }

# Run simulation for mean conditions
sim_result = run_dpf_simulation(config)

print("Simulation Results (mean conditions):")
print(f"  Peak current:   {sim_result['I_peak_kA']:.0f} kA")
print(f"  Rise time:      {sim_result['t_rise_us']:.2f} μs")
print(f"  Pinch time:     {sim_result['t_pinch_us']:.2f} μs")
print(f"  Neutron yield:  {sim_result['Y_n']:.2e}")

In [ ]:
# Plot simulated current waveform
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(sim_result['t'] * 1e6, sim_result['I'] / 1e3, 'b-', linewidth=2, label='Simulated current')
ax.axvline(sim_result['t_pinch_us'], color='red', linestyle='--', label=f'Pinch time: {sim_result["t_pinch_us"]:.2f} μs')
ax.axhline(0, color='k', linewidth=0.5)

ax.set_xlabel('Time (μs)', fontsize=12)
ax.set_ylabel('Current (kA)', fontsize=12)
ax.set_title('Simulated DPF Current Waveform', fontsize=14)
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 5])

plt.tight_layout()
plt.show()

## 6. Comparison with Experimental Data

Now we run simulations for each experimental shot and compare.

In [ ]:
# Run simulations for all experimental shots
sim_I_peak = []
sim_Y_n = []
sim_t_pinch = []

for shot in shots:
    shot_params = {
        'V0_kV': shot['V0_kV'],
        'pressure_mbar': shot['pressure_mbar']
    }
    config = create_simulation_config(exp_data['parameters'], shot_params)
    result = run_dpf_simulation(config)
    
    sim_I_peak.append(result['I_peak_kA'])
    sim_Y_n.append(result['Y_n'])
    sim_t_pinch.append(result['t_pinch_us'])

sim_I_peak = np.array(sim_I_peak)
sim_Y_n = np.array(sim_Y_n)
sim_t_pinch = np.array(sim_t_pinch)

print(f"Ran {len(shots)} simulations")

In [ ]:
# Comparison plots
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Peak current comparison
ax1 = axes[0, 0]
ax1.scatter(I_peak, sim_I_peak, alpha=0.6, edgecolors='k', linewidth=0.5)
lim = [min(I_peak.min(), sim_I_peak.min()) - 10, max(I_peak.max(), sim_I_peak.max()) + 10]
ax1.plot(lim, lim, 'r--', label='Perfect agreement')
ax1.set_xlabel('Experimental I_peak (kA)')
ax1.set_ylabel('Simulated I_peak (kA)')
ax1.set_title('Peak Current Comparison')
ax1.set_xlim(lim)
ax1.set_ylim(lim)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Calculate R² for current
r2_I = 1 - np.sum((I_peak - sim_I_peak)**2) / np.sum((I_peak - I_peak.mean())**2)
ax1.text(0.05, 0.95, f'R² = {r2_I:.3f}', transform=ax1.transAxes, verticalalignment='top', fontsize=12)

# Neutron yield comparison (log scale)
ax2 = axes[0, 1]
ax2.scatter(Y_n, sim_Y_n, alpha=0.6, edgecolors='k', linewidth=0.5, color='green')
lim_log = [1e6, 1e10]
ax2.plot(lim_log, lim_log, 'r--', label='Perfect agreement')
ax2.set_xlabel('Experimental Neutron Yield')
ax2.set_ylabel('Simulated Neutron Yield')
ax2.set_title('Neutron Yield Comparison')
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlim(lim_log)
ax2.set_ylim(lim_log)
ax2.legend()
ax2.grid(True, alpha=0.3, which='both')

# R² for log(yield)
log_Y_exp = np.log10(Y_n)
log_Y_sim = np.log10(sim_Y_n)
r2_Y = 1 - np.sum((log_Y_exp - log_Y_sim)**2) / np.sum((log_Y_exp - log_Y_exp.mean())**2)
ax2.text(0.05, 0.95, f'R² (log) = {r2_Y:.3f}', transform=ax2.transAxes, verticalalignment='top', fontsize=12)

# Residuals for current
ax3 = axes[1, 0]
residuals_I = (sim_I_peak - I_peak) / I_peak * 100
ax3.hist(residuals_I, bins=20, edgecolor='black', alpha=0.7)
ax3.axvline(0, color='red', linestyle='--')
ax3.axvline(residuals_I.mean(), color='green', linestyle='-', label=f'Mean: {residuals_I.mean():.1f}%')
ax3.set_xlabel('Relative Error in I_peak (%)')
ax3.set_ylabel('Count')
ax3.set_title('Current Prediction Residuals')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Residuals for yield (log scale)
ax4 = axes[1, 1]
residuals_Y = np.log10(sim_Y_n / Y_n)
ax4.hist(residuals_Y, bins=20, edgecolor='black', alpha=0.7, color='green')
ax4.axvline(0, color='red', linestyle='--')
ax4.axvline(residuals_Y.mean(), color='blue', linestyle='-', label=f'Mean: {residuals_Y.mean():.2f}')
ax4.set_xlabel('log₁₀(Simulated/Experimental) Yield')
ax4.set_ylabel('Count')
ax4.set_title('Yield Prediction Residuals')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Statistical Analysis of Agreement

We now perform rigorous statistical analysis of the model-data comparison.

In [ ]:
def calculate_validation_metrics(exp, sim, name='Quantity'):
    """
    Calculate comprehensive validation metrics.
    
    Parameters
    ----------
    exp : array-like
        Experimental values
    sim : array-like
        Simulated values
    name : str
        Name of the quantity being compared
        
    Returns
    -------
    dict
        Dictionary of validation metrics
    """
    exp = np.array(exp)
    sim = np.array(sim)
    
    # Basic statistics
    n = len(exp)
    
    # Correlation coefficient
    r, p_value = stats.pearsonr(exp, sim)
    
    # R-squared
    ss_res = np.sum((exp - sim)**2)
    ss_tot = np.sum((exp - exp.mean())**2)
    r2 = 1 - ss_res / ss_tot
    
    # Mean Absolute Error
    mae = np.mean(np.abs(exp - sim))
    
    # Root Mean Square Error
    rmse = np.sqrt(np.mean((exp - sim)**2))
    
    # Mean Absolute Percentage Error
    mape = np.mean(np.abs((exp - sim) / exp)) * 100
    
    # Bias (mean error)
    bias = np.mean(sim - exp)
    
    # Standard deviation of residuals
    std_residuals = np.std(sim - exp)
    
    return {
        'name': name,
        'n': n,
        'r': r,
        'p_value': p_value,
        'r2': r2,
        'mae': mae,
        'rmse': rmse,
        'mape': mape,
        'bias': bias,
        'std_residuals': std_residuals
    }

# Calculate metrics for each quantity
metrics_I = calculate_validation_metrics(I_peak, sim_I_peak, 'Peak Current (kA)')
metrics_Y = calculate_validation_metrics(np.log10(Y_n), np.log10(sim_Y_n), 'log₁₀(Neutron Yield)')
metrics_t = calculate_validation_metrics(t_pinch, sim_t_pinch, 'Pinch Time (μs)')

# Display results
print("="*70)
print("VALIDATION METRICS SUMMARY")
print("="*70)

for metrics in [metrics_I, metrics_Y, metrics_t]:
    print(f"\n{metrics['name']}")
    print("-" * 40)
    print(f"  Correlation (r):        {metrics['r']:.4f} (p = {metrics['p_value']:.2e})")
    print(f"  R²:                     {metrics['r2']:.4f}")
    print(f"  RMSE:                   {metrics['rmse']:.4f}")
    print(f"  Mean Absolute Error:    {metrics['mae']:.4f}")
    print(f"  Mean % Error:           {metrics['mape']:.1f}%")
    print(f"  Bias:                   {metrics['bias']:.4f}")
    print(f"  Std of Residuals:       {metrics['std_residuals']:.4f}")

### 7.1 Hypothesis Testing

We can test whether the model predictions are statistically consistent with observations.

In [ ]:
# Paired t-test: Are simulated values significantly different from experimental?
t_stat_I, p_val_I = stats.ttest_rel(I_peak, sim_I_peak)
t_stat_Y, p_val_Y = stats.ttest_rel(np.log10(Y_n), np.log10(sim_Y_n))
t_stat_t, p_val_t = stats.ttest_rel(t_pinch, sim_t_pinch)

print("Paired t-tests (H0: simulation = experiment)")
print("="*50)
print(f"Peak Current:   t = {t_stat_I:7.3f}, p = {p_val_I:.4f} {'*' if p_val_I < 0.05 else ''}")
print(f"Neutron Yield:  t = {t_stat_Y:7.3f}, p = {p_val_Y:.4f} {'*' if p_val_Y < 0.05 else ''}")
print(f"Pinch Time:     t = {t_stat_t:7.3f}, p = {p_val_t:.4f} {'*' if p_val_t < 0.05 else ''}")
print("\n* indicates statistically significant difference at α=0.05")

## 8. Identifying Model Limitations

Let's analyze where and why the model might be failing.

In [ ]:
# Residual analysis - look for patterns
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Residuals vs voltage
ax1 = axes[0, 0]
ax1.scatter(V0, residuals_I, alpha=0.6)
z = np.polyfit(V0, residuals_I, 1)
p = np.poly1d(z)
ax1.plot(V0, p(V0), 'r-', label=f'Trend: slope = {z[0]:.2f}')
ax1.axhline(0, color='k', linestyle='--')
ax1.set_xlabel('Charging Voltage (kV)')
ax1.set_ylabel('Current Residual (%)')
ax1.set_title('Residuals vs Voltage')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Residuals vs pressure
ax2 = axes[0, 1]
ax2.scatter(pressure, residuals_Y, alpha=0.6, color='green')
z = np.polyfit(pressure, residuals_Y, 1)
p = np.poly1d(z)
ax2.plot(pressure, p(pressure), 'r-', label=f'Trend: slope = {z[0]:.2f}')
ax2.axhline(0, color='k', linestyle='--')
ax2.set_xlabel('Pressure (mbar)')
ax2.set_ylabel('Yield Residual (log₁₀)')
ax2.set_title('Yield Residuals vs Pressure')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Q-Q plot for normality of residuals
ax3 = axes[1, 0]
stats.probplot(residuals_I, dist="norm", plot=ax3)
ax3.set_title('Q-Q Plot: Current Residuals')
ax3.grid(True, alpha=0.3)

# Yield residual vs current
ax4 = axes[1, 1]
ax4.scatter(I_peak, residuals_Y, alpha=0.6, color='purple')
ax4.axhline(0, color='k', linestyle='--')
ax4.set_xlabel('Peak Current (kA)')
ax4.set_ylabel('Yield Residual (log₁₀)')
ax4.set_title('Yield Residuals vs Peak Current')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey observations:")
print("- If residuals show trends with input parameters, the model may be missing physics")
print("- Non-normal residuals suggest systematic errors")
print("- Large scatter indicates stochastic processes not captured by the model")

## 9. Exercises

### Exercise 1: Sensitivity Analysis

Vary a single model parameter and observe its effect on validation metrics.

In [ ]:
# Exercise 1: Your solution here
# ==============================

# TODO: Vary the circuit inductance by ±20% and recalculate
# How does this affect R² for peak current?

# inductance_variations = [0.8, 0.9, 1.0, 1.1, 1.2]  # multipliers
# r2_values = []
# 
# for L_mult in inductance_variations:
#     # Modify config and run simulations
#     # Calculate R² for each case
#     pass
#
# Plot R² vs inductance multiplier

### Exercise 2: Model Calibration

Use optimization to find the best-fit model parameters.

In [ ]:
# Exercise 2: Your solution here
# ==============================

# TODO: Use scipy.optimize.minimize to find optimal k_yield parameter
# that minimizes RMSE between simulated and experimental neutron yields

# def objective(params):
#     k_yield = params[0]
#     # Run simulations with this k_yield
#     # Return RMSE between log(sim) and log(exp)
#     pass
#
# result = minimize(objective, x0=[2e-17], method='Nelder-Mead')
# optimal_k = result.x[0]

### Exercise 3: Cross-Validation

Implement k-fold cross-validation to assess model generalization.

In [ ]:
# Exercise 3: Your solution here
# ==============================

# TODO: Split data into 5 folds
# Train (calibrate k_yield) on 4 folds, test on 1 fold
# Report mean and std of R² across folds

# from sklearn.model_selection import KFold
# 
# kf = KFold(n_splits=5, shuffle=True, random_state=42)
# r2_scores = []
# 
# for train_idx, test_idx in kf.split(shots):
#     # Calibrate on training set
#     # Evaluate on test set
#     pass

## 10. Summary

In this tutorial, you learned:

1. **Data loading**: How to structure and explore experimental DPF data
2. **Simulation setup**: Configuring dpf2 to match experimental conditions
3. **Comparison methods**: Visual and statistical comparison of results
4. **Validation metrics**: R², RMSE, MAPE, and hypothesis testing
5. **Residual analysis**: Identifying model limitations and systematic errors

### Key Takeaways

| Aspect | Key Point |
|--------|----------|
| Correlation | High r doesn't guarantee good predictions |
| Bias | Check for systematic over/under-prediction |
| Scatter | Large variance indicates missing physics |
| Trends | Residual patterns reveal model deficiencies |
| Uncertainty | Report confidence intervals, not just means |

## 11. Further Reading

- AIAA Guide for Verification and Validation of Computational Fluid Dynamics Simulations (AIAA G-077-1998)
- Oberkampf, W.L. & Roy, C.J. "Verification and Validation in Scientific Computing" (Cambridge, 2010)
- Lee, S. et al. "Neutron Yield Saturation in Plasma Focus" (Applied Physics Letters, 1988)